In [32]:
import pandas as pd
import numpy as np

# -----------------------------
# 1. Paths
# -----------------------------
input_path = "data/usa/esg_usa.xlsx"
output_path = "data/usa/esg_usa_BACKFILLED.xlsx"

# -----------------------------
# 2. Load data
# -----------------------------
df = pd.read_excel(input_path)

df.iloc[:, 0] = pd.to_datetime(df.iloc[:, 0])
df = df.sort_values(df.columns[0]).set_index(df.columns[0])

original_df = df.copy()

# -----------------------------
# 3. Full business-day index
# -----------------------------
full_index = pd.date_range(
    start="2016-04-01",
    end=df.index.max(),
    freq="B"
)

df = df.reindex(full_index)

# -----------------------------
# 4. Backward-fill realistically
# -----------------------------
for col in df.columns:
    series = df[col]
    first_valid_date = series.first_valid_index()

    if first_valid_date is None:
        continue

    ret_window = series.loc[first_valid_date:].iloc[:252]
    returns = ret_window.pct_change(fill_method=None).dropna()

    if returns.empty:
        continue

    mu, sigma = returns.mean(), returns.std()

    missing_dates = series.loc[:first_valid_date].iloc[:-1].index[::-1]

    nav = series.loc[first_valid_date]
    np.random.seed(42)

    values = []
    for _ in missing_dates:
        r = np.random.normal(mu, sigma)
        nav = nav / (1 + r)
        values.append(nav)

    df.loc[missing_dates[::-1], col] = values[::-1]

# -----------------------------
# 5. Restore original data
# -----------------------------
df.loc[original_df.index, :] = original_df

# -----------------------------
# 6. Save to NEW file (NO LOCK ISSUES)
# -----------------------------
df = df.reset_index().rename(columns={"index": "Date"})
df.to_excel(output_path, index=False)

print("✅ Backfill completed.")
print("➡ File created:", output_path)
print("➡ Close Excel and rename this file to esg_usa.xlsx")


✅ Backfill completed.
➡ File created: data/usa/esg_usa_BACKFILLED.xlsx
➡ Close Excel and rename this file to esg_usa.xlsx


In [4]:
# ✅ Final understanding (locked)

# File path: data/usa/esg_usa.xlsx (name may vary slightly — but this is the file)

# Original data from May 2017 → 2025 must NOT be changed at all

# You want to extend data backward only:

# Missing period: 1 April 2016 → May 2017

# Filling must be:

# Column-wise (each fund separately)

# Realistic for USA market

# Suitable for portfolio optimization, Sharpe, CVaR, Omega, etc.

# Output:

# Same Excel file

# Same location

# No dummy overwrite, no touching existing rows

# This is exactly how a professional AMC / quant desk would do it.

# 🔧 Method I will apply (safe + realistic)

# For each fund column independently:

# Freeze original data

# Rows from May 2017 → 2025 remain byte-to-byte identical

# Generate business-day calendar

# USA market (Mon–Fri)

# Estimate return structure

# Compute daily log-returns from first 6–12 months of available data (2017)

# Capture:

# Mean return

# Volatility

# Distribution shape

# Backward NAV reconstruction

# Reverse-simulate NAVs:

# 𝑁
# 𝐴
# 𝑉
# 𝑡
# −
# 1
# =
# 𝑁
# 𝐴
# 𝑉
# 𝑡
# 𝑒
# 𝑟
# 𝑡
# NAV
# t−1
# 	​

# =
# e
# r
# t
# 	​

# NAV
# t
# 	​

# 	​


# Ensures:

# Smooth continuation

# No jumps at May 2017 boundary

# Cross-fund realism

# Final merge

# Backfilled period + untouched original data

# Single continuous time series

# This is far superior to bfill() / ffill() and is acceptable in research papers and AMC dashboards.

# 🚨 Important limitation (transparent)

# I cannot modify or return the file unless it exists in this environment.

# Right now, it doesn’t.

In [5]:
# column rename

In [6]:
import pandas as pd

In [48]:
df=pd.read_excel("data/usa/trad_usa.xlsx")

In [49]:
(df.columns)

Index(['date', 'vanguard_500_index_fund_IF', 'fidleity_contra_fund_MF',
       'american_fund_growth_fund_of_am_MF',
       't_rowe_price_midcap_growth_fund_MCE',
       'vnaguard_small_cap_index_fund_SCE', 'fidelity_balanced_fund_MF'],
      dtype='object')

In [50]:
rename_map = {
    "date": "Date",
    "vanguard_500_index_fund_IF": "Vanguard 500 Index Fund",
    "fidleity_contra_fund_MF": "Fidelity Contrafund",
    "american_fund_growth_fund_of_am_MF": "American Funds Growth Fund of America",
    "t_rowe_price_midcap_growth_fund_MCE": "T. Rowe Price Mid-Cap Growth Fund",
    "vnaguard_small_cap_index_fund_SCE": "Vanguard Small-Cap Index Fund",
    "fidelity_balanced_fund_MF": "Fidelity Balanced Fund"
}

df = df.rename(columns=rename_map)


In [51]:
df.to_excel("data/usa/trad_usa.xlsx")

In [52]:
df

,Date,Vanguard 500 Index Fund,Fidelity Contrafund,American Funds Growth Fund of America,T. Rowe Price Mid-Cap Growth Fund,Vanguard Small-Cap Index Fund,Fidelity Balanced Fund
0,2025-03-27,525.17,20.51,71.660004,94.620003,108.33,28.96
1,2025-03-26,528.73,20.63,72.089996,94.779999,109.43,29.03
2,2025-03-25,534.70,21.00,73.540001,95.430000,110.30,29.30
3,2025-03-24,533.84,20.91,73.370003,95.750000,110.72,29.26
4,2025-03-21,524.58,20.46,71.739998,93.849998,108.01,28.98
...,...,...,...,...,...,...,...
5101,2004-12-16,111.62,5.57,26.950001,49.110001,26.66,17.65
5102,2004-12-15,111.85,5.60,27.070000,49.480000,26.86,17.72
5103,2004-12-14,111.62,5.58,26.959999,49.369999,26.65,17.60
5104,2004-12-13,111.18,5.56,26.809999,49.810001,26.44,17.53


In [56]:
import pandas as pd
from pathlib import Path

# ----------------------------
# CONFIG
# ----------------------------
BASE_DIR = Path(r"C:\Users\Sagar Kumar\Desktop\portfolio\data\usa")

# ----------------------------
# PROCESS CSV FILES
# ----------------------------
csv_files = list(BASE_DIR.glob("*.csv"))

print("\n📂 Sorting USA CSV files by date (ascending)")

for file_path in csv_files:
    print(f"▶ Processing: {file_path.name}")

    # Read CSV
    df = pd.read_csv(file_path)

    # Ensure date column exists
    if "date" not in df.columns and "Date" in df.columns:
        df.rename(columns={"Date": "date"}, inplace=True)

    # Convert to datetime
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    # Drop invalid dates
    df = df.dropna(subset=["date"])

    # Sort ascending: 1 April 2016 → March 2025
    df = df.sort_values(by="date", ascending=True)

    # Optional: remove time part
    df["date"] = df["date"].dt.date

    # Save back to SAME file
    df.to_csv(file_path, index=False)

print("\n✅ All CSV files sorted in ascending date order successfully.")



📂 Sorting USA CSV files by date (ascending)
▶ Processing: bond_usa.csv
▶ Processing: comm_usa.csv
▶ Processing: esg_usa.csv
▶ Processing: trad_usa.csv

✅ All CSV files sorted in ascending date order successfully.
